# Session 2: Live demo, the computer finds the rule
Same made-up data as Challenge C. Run each cell in order (Shift + Enter).

## 1. The data

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

train = pd.DataFrame({
    "food":      ["Maize", "Maize", "Maize", "Maize", "Rice", "Rice", "Swordfish", "Swordfish", "Apple", "Apple", "Maize", "Rice"],
    "origin":    ["non-EU", "non-EU", "EU", "EU", "non-EU", "EU", "non-EU", "EU", "EU", "non-EU", "non-EU", "non-EU"],
    "season":    ["summer", "winter", "summer", "winter", "summer", "winter", "summer", "winter", "summer", "winter", "summer", "winter"],
    "retested":  ["yes", "yes", "no", "no", "no", "no", "yes", "yes", "no", "no", "yes", "no"],
    "non_compliant": ["yes", "yes", "no", "no", "no", "no", "yes", "yes", "no", "no", "yes", "no"],
})
test = pd.DataFrame({
    "food":   ["Maize", "Swordfish", "Rice", "Maize", "Apple", "Maize"],
    "origin": ["non-EU", "EU", "EU", "EU", "non-EU", "non-EU"],
    "season": ["summer", "summer", "summer", "summer", "summer", "winter"],
    "non_compliant": ["yes", "yes", "no", "yes", "no", "no"],
})
print(len(train), "training batches,", len(test), "test batches")

12 training batches, 6 test batches


## 2. Train with every column, "retested" included

In [2]:
cols = ["food", "origin", "season", "retested"]
X = pd.get_dummies(train[cols])
y = train["non_compliant"]

model = DecisionTreeClassifier(random_state=0)
model.fit(X, y)

names = list(X.columns)
print(export_text(model, feature_names=names))
print("Training accuracy:", model.score(X, y))

|--- retested_yes <= 0.50
|   |--- class: no
|--- retested_yes >  0.50
|   |--- class: yes

Training accuracy: 1.0


## 3. Leakage: "retested" does not exist yet for new batches. Drop it and train again

In [3]:
cols = ["food", "origin", "season"]
X = pd.get_dummies(train[cols])

model = DecisionTreeClassifier(random_state=0)
model.fit(X, y)

names = list(X.columns)
print(export_text(model, feature_names=names))

|--- food_Swordfish <= 0.50
|   |--- food_Maize <= 0.50
|   |   |--- class: no
|   |--- food_Maize >  0.50
|   |   |--- origin_EU <= 0.50
|   |   |   |--- class: yes
|   |   |--- origin_EU >  0.50
|   |   |   |--- class: no
|--- food_Swordfish >  0.50
|   |--- class: yes



## 4. Predict the 6 new batches

In [4]:
X_test = pd.get_dummies(test[cols]).reindex(columns=X.columns, fill_value=False)
test["prediction"] = model.predict(X_test)
print(test[["food", "origin", "season", "prediction", "non_compliant"]])
print("Correct:", (test["prediction"] == test["non_compliant"]).sum(), "out of", len(test))

        food  origin  season prediction non_compliant
0      Maize  non-EU  summer        yes           yes
1  Swordfish      EU  summer        yes           yes
2       Rice      EU  summer         no            no
3      Maize      EU  summer         no           yes
4      Apple  non-EU  summer         no            no
5      Maize  non-EU  winter        yes            no
Correct: 4 out of 6
